<a href="https://colab.research.google.com/github/AntonDozhdikov/Demography_migration/blob/main/MARL_Kursk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# KURSK DEMOGRAPHIC MARL — STAGE 1
# Multi-Agent Reinforcement Learning for Regional Demographic & Migration Policy
# Kursk Oblast, Russian Federation | 1991–2025 historical + 2026–2050 forecast
#
# Architecture:
#   ├── Section 0:  Colab setup & reproducibility
#   ├── Section 1:  Data loading, EDA & preprocessing
#   ├── Section 2:  World-model training (transition model)
#   ├── Section 3:  Custom multi-agent Gym environment
#   ├── Section 4:  MARL agents (MADDPG + MATD3 +  baselines)
#   ├── Section 5:  Training loop with full logging
#   ├── Section 6:  Evaluation: MARL vs Baseline trajectories
#   ├── Section 7:  Policy analysis & action preference maps
#   ├── Section 8:  Publication-grade visualisations
#   └── Section 9:  Export — CSV, JSON, plots for scientific report
#
# Agents (8 total):
#   INSTITUTIONAL (4): RegionalGov, HealthcareSystem, EducationInfra, MigrationPolicy
#   INDIVIDUAL    (4): Women (reproductive), Households, Families, Migrants
#
# Requirements: torch, gymnasium, numpy, pandas, matplotlib, seaborn,
#               scikit-learn, scipy, openpyxl, tqdm, tensorboard
# =============================================================================


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 — COLAB SETUP & REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os, json, pickle, warnings, time
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

def _install(pkg):
    """Silently install a missing package (Colab-safe)."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Ensure required packages are available
REQUIRED = {
    "torch": "torch",
    "gymnasium": "gymnasium",
    "openpyxl": "openpyxl",
    "tensorboard": "tensorboard",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "tqdm": "tqdm",
}
for imp, pkg in REQUIRED.items():
    try:
        __import__(imp)
    except ImportError:
        print(f"Installing {pkg}...")
        _install(pkg)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")          # headless rendering — works in Colab & local
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
from collections import deque, defaultdict, OrderedDict
from typing import Dict, List, Tuple, Optional, Any
from tqdm.auto import tqdm, trange
from copy import deepcopy

# ── Reproducibility seed ─────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Device: {DEVICE} | PyTorch {torch.__version__} | Seed {SEED}")

# ── Output directories ────────────────────────────────────────────────────────
BASE_DIR   = Path("kursk_marl_output")
PLOT_DIR   = BASE_DIR / "plots"
LOG_DIR    = BASE_DIR / "logs"
MODEL_DIR  = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "report_data"

for d in [PLOT_DIR, LOG_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[Setup] Output root: {BASE_DIR.resolve()}")

# ── Experiment config (single source of truth) ────────────────────────────────
CFG = {
    # ── Data
    "data_file":         "Kursk_state_space_1991_2050.xlsx",
    "sheet":             "state_space",
    "year_col":          "Год",
    "hist_end":          2025,
    "forecast_end":      2050,
    "train_end":         2012,       # train/val/test temporal split
    "val_end":           2018,
    # ── Environment
    "dt":                1,          # timestep = 1 year
    "horizon":           25,         # 2026–2050
    "n_scenarios":       30,         # stochastic rollouts per eval
    "shock_std":         0.02,       # Gaussian noise on transitions
    # ── Agents
    "n_agents":          8,
    "n_infra_agents":    4,
    "n_indiv_agents":    4,
    # ── MADDPG hyperparameters
    "actor_lr":          1e-3,
    "critic_lr":         1e-3,
    "gamma":             0.95,
    "tau":               0.005,       # soft target update
    "buffer_size":       50_000,
    "batch_size":        256,
    "hidden_dim":        256,
    "n_episodes":        600,
    "max_steps":         25,
    "warmup_steps":      500,
    # ── Reward weights (α vector in paper)
    "reward_w": {
        "pop_growth":       0.20,
        "tfr":              0.25,
        "net_migration":    0.15,
        "nat_increase":     0.15,
        "housing":          0.10,
        "healthcare":       0.10,
        "cost_penalty":     -0.05,
    },
    "seed":              SEED,
    "device":            str(DEVICE),
    "timestamp":         datetime.now().isoformat(),
}

with open(REPORT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)

print("[Setup] Config saved → experiment_config.json")

[Setup] Device: cuda | PyTorch 2.10.0+cu128 | Seed 42
[Setup] Output root: /content/kursk_marl_output
[Setup] Config saved → experiment_config.json


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING, EDA & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 1 — DATA LOADING & PREPROCESSING")
print("="*70)

# ── 1.1 Load raw dataset ─────────────────────────────────────────────────────

def load_state_space(path: str, sheet: str = "state_space") -> pd.DataFrame:
    """
    Load the Kursk state-space dataset from Excel.
    Returns a DataFrame indexed by year with all numeric columns.
    The 'Тип половозрастной структуры (Сундберг)' column is ordinally encoded.
    """
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.rename(columns={df.columns[0]: "year"})
    df = df.set_index("year").sort_index()

    # Ordinal encode the Sunberg population type
    sunberg_map = {"прогрессивный": 2, "стационарный": 1, "регрессивный": 0}
    sun_col = "Тип половозрастной структуры (Сундберг)"
    if sun_col in df.columns:
        df[sun_col] = df[sun_col].map(sunberg_map).fillna(0).astype(int)

    # Force all columns to numeric, coerce errors → NaN
    df = df.apply(pd.to_numeric, errors="coerce")

    # Forward-fill small gaps (≤2 years)
    df = df.fillna(method="ffill", limit=2)
    df = df.fillna(method="bfill", limit=2)

    return df


# Try to load from working directory (Colab upload) or current path
for candidate in [CFG["data_file"], f"../{CFG['data_file']}", f"/content/{CFG['data_file']}"]:
    if Path(candidate).exists():
        RAW_DF = load_state_space(candidate, CFG["sheet"])
        print(f"[Data] Loaded from: {candidate}")
        break
else:
    raise FileNotFoundError(
        f"Dataset '{CFG['data_file']}' not found. "
        "Upload to Colab working directory or adjust the path."
    )

print(f"[Data] Shape: {RAW_DF.shape} | Years: {RAW_DF.index.min()}–{RAW_DF.index.max()}")
print(f"[Data] Columns ({len(RAW_DF.columns)}): {list(RAW_DF.columns[:5])} ...")

# ── 1.2 Feature taxonomy ─────────────────────────────────────────────────────
# We group the 55+ columns into semantic clusters used by different agents.

FEATURE_GROUPS: Dict[str, List[str]] = {}

_all_cols = list(RAW_DF.columns)

# Helper: find columns that contain any keyword (case-insensitive)
def _cols(*keywords):
    return [c for c in _all_cols if any(k.lower() in c.lower() for k in keywords)]

FEATURE_GROUPS["population_core"] = _cols(
    "Численность населения всего",
    "Численность женщин 15-49",
    "Доля женщин 15-49",
    "Численность мужчин 15-49",
    "Соотношение мужчин и женщин",
    "Численность населения моложе",
    "Численность населения старше",
    "Тип половозрастной",
)
FEATURE_GROUPS["fertility"] = _cols(
    "рождений", "СКР", "Средний возраст матери",
    "Доля рождений в браке", "Доля рождений вне брака",
    "брачности", "разводимости", "браков к числу разводов",
)
FEATURE_GROUPS["migration"] = _cols(
    "миграционного прироста", "мигрантов моложе",
    "мигрантов трудоспособного", "мигрантов старше",
    "естественного прироста",
)
FEATURE_GROUPS["reproductive_health"] = _cols(
    "аборт", "бесплодие", "бесплодием", "ЭКО",
    "родов после ЭКО", "рождённых после ЭКО",
)
FEATURE_GROUPS["healthcare_infra"] = _cols(
    "педиатр", "акушер", "неонатолог", "врачей",
    "больничных коек", "Обеспеченность койками",
    "инвалидов",
)
FEATURE_GROUPS["housing"] = _cols(
    "площадь жилья", "площадь благоустроенного",
    "Доля благоустроенного",
    "молодых семей", "многодетных семей",
)
FEATURE_GROUPS["education_childcare"] = _cols(
    "ДОУ", "дошкольным", "ГПД",
    "занятости женщин с детьми",
)

# Flatten to get the state vector columns (all features used in RL)
STATE_COLS: List[str] = []
for grp_cols in FEATURE_GROUPS.values():
    for c in grp_cols:
        if c not in STATE_COLS and c in RAW_DF.columns:
            STATE_COLS.append(c)

# Remove duplicates while preserving order
STATE_COLS = list(OrderedDict.fromkeys(STATE_COLS))
STATE_DIM = len(STATE_COLS)
print(f"[Data] State vector dimension: {STATE_DIM}")

# ── 1.3 Split historical / forecast ──────────────────────────────────────────

HIST_DF = RAW_DF.loc[1991:CFG["hist_end"], STATE_COLS].copy()
FORE_DF = RAW_DF.loc[CFG["hist_end"]+1:CFG["forecast_end"], STATE_COLS].copy()

TRAIN_DF = HIST_DF.loc[:CFG["train_end"]]
VAL_DF   = HIST_DF.loc[CFG["train_end"]+1:CFG["val_end"]]
TEST_DF  = HIST_DF.loc[CFG["val_end"]+1:]

print(f"[Data] Train: {TRAIN_DF.index[0]}–{TRAIN_DF.index[-1]} ({len(TRAIN_DF)} yr)")
print(f"[Data] Val  : {VAL_DF.index[0]}–{VAL_DF.index[-1]} ({len(VAL_DF)} yr)")
print(f"[Data] Test : {TEST_DF.index[0]}–{TEST_DF.index[-1]} ({len(TEST_DF)} yr)")
print(f"[Data] Fore : {FORE_DF.index[0]}–{FORE_DF.index[-1]} ({len(FORE_DF)} yr)")

# ── 1.4 Normalisation — fit on TRAIN, apply everywhere ──────────────────────

SCALER = StandardScaler()
SCALER.fit(TRAIN_DF.values)

def to_scaled(df: pd.DataFrame) -> np.ndarray:
    return SCALER.transform(df[STATE_COLS].values)

def from_scaled(arr: np.ndarray) -> np.ndarray:
    return SCALER.inverse_transform(arr)

TRAIN_SC = to_scaled(TRAIN_DF)
VAL_SC   = to_scaled(VAL_DF)
TEST_SC  = to_scaled(TEST_DF)
HIST_SC  = to_scaled(HIST_DF)
FORE_SC  = to_scaled(FORE_DF)

# ── 1.5 EDA summary ──────────────────────────────────────────────────────────

eda_stats = HIST_DF.describe().T
eda_stats["cv"] = eda_stats["std"] / eda_stats["mean"].abs()
eda_stats.to_csv(REPORT_DIR / "eda_descriptive_stats.csv")

# Compute year-on-year % change for historical period
yoy_pct = HIST_DF.pct_change().mul(100).round(3)
yoy_pct.to_csv(REPORT_DIR / "historical_yoy_pct_change.csv")

# Correlation matrix (Spearman — robust to outliers)
corr_matrix = HIST_DF[STATE_COLS].corr(method="spearman")
corr_matrix.to_csv(REPORT_DIR / "spearman_correlation_matrix.csv")

print("[Data] EDA files saved.")

# ── 1.6 EDA visualisation — population & fertility overview ──────────────────

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Курская область — ключевые демографические показатели 1991–2025",
             fontsize=15, fontweight="bold", y=1.01)

panel_vars = [
    ("Численность населения всего",       "Общая численность населения"),
    ("СКР (всего)",                        "СКР (суммарный коэффициент рождаемости)"),
    ("Коэффициент естественного прироста (на 1000)",
                                           "Коэффициент естественного прироста (на 1000)"),
    ("Коэффициент миграционного прироста (на 10000)",
                                           "Коэффициент миграционного прироста (на 10000)"),
    ("Число абортов на 100 родов",         "Аборты на 100 родов"),
    ("Число циклов ЭКО",                   "Число циклов ЭКО"),
    ("Общая площадь жилья на 1 жителя (кв. м)",
                                           "Жильё на 1 жителя, кв. м"),
    ("Обеспеченность местами в ДОУ (на 1000 детей)",
                                           "Обеспеченность ДОУ (на 1000 детей)"),
    ("Численность врачей",                 "Численность врачей"),
]

colors = plt.cm.tab10.colors
for ax, (col, title) in zip(axes.flat, panel_vars):
    if col in HIST_DF.columns:
        years = HIST_DF.index.values
        vals  = HIST_DF[col].values
        ax.plot(years, vals, color=colors[0], linewidth=2)
        ax.fill_between(years, vals, alpha=0.15, color=colors[0])
        ax.axvline(CFG["train_end"], color="gray", linestyle="--", alpha=0.7, linewidth=1)
        ax.axvline(CFG["val_end"],   color="gray", linestyle=":",  alpha=0.7, linewidth=1)
        ax.set_title(title, fontsize=9, fontweight="bold")
        ax.set_xlabel("Год", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(PLOT_DIR / "01_eda_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 01_eda_overview.png saved.")

# ── Correlation heatmap (top 15 features) ────────────────────────────────────
top_feats = corr_matrix.abs().sum().nlargest(15).index.tolist()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix.loc[top_feats, top_feats], dtype=bool))
sns.heatmap(corr_matrix.loc[top_feats, top_feats],
            mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Матрица корреляций Спирмена (топ-15 признаков)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 02_correlation_heatmap.png saved.")



SECTION 1 — DATA LOADING & PREPROCESSING
[Data] Loaded from: Kursk_state_space_1991_2050.xlsx
[Data] Shape: (60, 53) | Years: 1991–2050
[Data] Columns (53): ['Численность населения всего', 'Численность женщин 15-49 лет', 'Доля женщин 15-49 лет в общей численности женщин', 'Численность мужчин 15-49 лет', 'Соотношение мужчин и женщин 15-49 лет (мужчин на 1 женщину)'] ...
[Data] State vector dimension: 53
[Data] Train: 1991–2012 (22 yr)
[Data] Val  : 2013–2018 (6 yr)
[Data] Test : 2019–2025 (7 yr)
[Data] Fore : 2026–2050 (25 yr)
[Data] EDA files saved.
[Plot] 01_eda_overview.png saved.
[Plot] 02_correlation_heatmap.png saved.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — WORLD MODEL (TRANSITION MODEL)
# ─────────────────────────────────────────────────────────────────────────────
# We train a neural transition model  f: (s_t, a_t) → s_{t+1}
# using historical data 1991–2012 (train).
# Two architectures: (a) MLP ensemble; (b) per-feature GBM (interpretable baseline).
# The RL environment will call the neural world model for rollouts.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 2 — WORLD MODEL TRAINING")
print("="*70)

# ── 2.1 Neural transition model ───────────────────────────────────────────────

class TransitionNet(nn.Module):
    """
    Probabilistic MLP transition model:
        μ, log_σ = TransitionNet(s_t)   [actions embedded as part of s_t via concatenation]
    Returns a Normal distribution over next state.
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        in_dim = state_dim + action_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, hidden // 2), nn.SiLU(),
        )
        self.mu_head     = nn.Linear(hidden // 2, state_dim)
        self.logvar_head = nn.Linear(hidden // 2, state_dim)

    def forward(self, s: torch.Tensor, a: torch.Tensor):
        x   = torch.cat([s, a], dim=-1)
        h   = self.net(x)
        mu  = self.mu_head(h)
        lv  = self.logvar_head(h).clamp(-10, 2)
        return mu, lv

    def predict(self, s: torch.Tensor, a: torch.Tensor,
                deterministic: bool = False) -> torch.Tensor:
        mu, lv = self.forward(s, a)
        if deterministic:
            return s + mu                        # residual connection
        eps = torch.randn_like(mu)
        return s + mu + eps * (0.5 * lv).exp()  # residual + stochastic


# Ensemble of 5 transition models (reduces epistemic uncertainty)
ENSEMBLE_SIZE = 5
ACTION_DIM_WORLD = 16  # placeholder action dimension for world model pre-training
                        # (actions are not yet defined; we use zero-action = baseline)

world_ensemble: List[TransitionNet] = [
    TransitionNet(STATE_DIM, ACTION_DIM_WORLD).to(DEVICE)
    for _ in range(ENSEMBLE_SIZE)
]
world_optimizers = [
    optim.AdamW(m.parameters(), lr=5e-4, weight_decay=1e-4)
    for m in world_ensemble
]

# ── Build supervised training sequences from historical data ─────────────────
# Input:  s_t (scaled), zeros for actions (no control during history)
# Target: s_{t+1} - s_t  (learn residual = Δstate)

def build_transition_dataset(scaled_arr: np.ndarray, action_dim: int) -> Tuple:
    """Convert scaled time series into (S_t, A_t, S_{t+1}) triplets."""
    T = len(scaled_arr)
    S   = scaled_arr[:-1]                               # (T-1, state_dim)
    S1  = scaled_arr[1:]                                # (T-1, state_dim)
    A   = np.zeros((T - 1, action_dim), dtype=np.float32)  # zero = no intervention
    return (
        torch.tensor(S,  dtype=torch.float32).to(DEVICE),
        torch.tensor(A,  dtype=torch.float32).to(DEVICE),
        torch.tensor(S1, dtype=torch.float32).to(DEVICE),
    )

S_tr, A_tr, S1_tr = build_transition_dataset(HIST_SC[:len(TRAIN_SC)], ACTION_DIM_WORLD)
S_va, A_va, S1_va = build_transition_dataset(HIST_SC[len(TRAIN_SC):len(TRAIN_SC)+len(VAL_SC)], ACTION_DIM_WORLD)

# ── Training loop — ensemble ──────────────────────────────────────────────────

WORLD_EPOCHS = 500
best_val_loss = [float("inf")] * ENSEMBLE_SIZE
world_train_losses: List[List[float]] = [[] for _ in range(ENSEMBLE_SIZE)]
world_val_losses:   List[List[float]] = [[] for _ in range(ENSEMBLE_SIZE)]

print(f"[WorldModel] Training {ENSEMBLE_SIZE}-member ensemble, {WORLD_EPOCHS} epochs...")

for epoch in trange(WORLD_EPOCHS, desc="WorldModel", ncols=80):
    for idx, (model, opt) in enumerate(zip(world_ensemble, world_optimizers)):
        model.train()
        mu, lv  = model(S_tr, A_tr)
        delta_tr = S1_tr - S_tr
        # Negative log-likelihood of a Gaussian: 0.5*(log_var + (x-mu)^2/var)
        nll = 0.5 * (lv + (delta_tr - mu).pow(2) / lv.exp()).mean()
        opt.zero_grad()
        nll.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        world_train_losses[idx].append(nll.item())

        # Validation
        model.eval()
        with torch.no_grad():
            mu_v, lv_v = model(S_va, A_va)
            delta_va   = S1_va - S_va
            nll_v = 0.5 * (lv_v + (delta_va - mu_v).pow(2) / lv_v.exp()).mean()
            world_val_losses[idx].append(nll_v.item())

            if nll_v.item() < best_val_loss[idx]:
                best_val_loss[idx] = nll_v.item()
                torch.save(model.state_dict(), MODEL_DIR / f"world_model_{idx}.pt")

print(f"[WorldModel] Best val NLL per member: {[f'{v:.4f}' for v in best_val_loss]}")

# Reload best checkpoints
for idx, model in enumerate(world_ensemble):
    model.load_state_dict(torch.load(MODEL_DIR / f"world_model_{idx}.pt", map_location=DEVICE))
    model.eval()

# ── 2.2 World-model evaluation on TEST set ───────────────────────────────────

S_te, A_te, S1_te = build_transition_dataset(HIST_SC[len(TRAIN_SC)+len(VAL_SC):], ACTION_DIM_WORLD)

with torch.no_grad():
    preds_te = torch.stack([m.predict(S_te, A_te, deterministic=True) for m in world_ensemble])
    pred_mean = preds_te.mean(0).cpu().numpy()   # ensemble mean prediction
    pred_std  = preds_te.std(0).cpu().numpy()    # epistemic uncertainty

# Inverse transform to original scale for reporting
true_orig = from_scaled(S1_te.cpu().numpy())
pred_orig = from_scaled(pred_mean)

# Per-feature MAPE and R²
wm_metrics = {}
for i, col in enumerate(STATE_COLS):
    y_true = true_orig[:, i]
    y_pred = pred_orig[:, i]
    if np.std(y_true) < 1e-8:
        continue
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    wm_metrics[col] = {"MAPE_%": round(mape, 3), "R2": round(r2, 4)}

wm_df = pd.DataFrame(wm_metrics).T.sort_values("MAPE_%")
wm_df.to_csv(REPORT_DIR / "world_model_test_metrics.csv")
print(f"[WorldModel] Median MAPE on test: {wm_df['MAPE_%'].median():.2f}%")
print(f"[WorldModel] Median R²   on test: {wm_df['R2'].median():.4f}")

# ── 2.3 World-model training curve plot ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx in range(ENSEMBLE_SIZE):
    axes[0].plot(world_train_losses[idx], alpha=0.7, label=f"M{idx}")
    axes[1].plot(world_val_losses[idx],   alpha=0.7, label=f"M{idx}")
for ax, title in zip(axes, ["Train NLL", "Val NLL"]):
    ax.set_title(f"World Model — {title}", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "03_world_model_training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 03_world_model_training_curves.png saved.")

# ── 2.4 World-model prediction vs actual (test period) ───────────────────────
n_show = min(6, len(STATE_COLS))
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("World Model — прогноз vs факт (тестовый период)", fontsize=13, fontweight="bold")
years_te = TEST_DF.index[1:]
for ax, col in zip(axes.flat, STATE_COLS[:n_show]):
    i = STATE_COLS.index(col)
    ax.plot(years_te, true_orig[:, i], "b-", linewidth=2, label="Факт")
    ax.plot(years_te, pred_orig[:, i], "r--", linewidth=2, label="Модель")
    ax.fill_between(years_te,
                    from_scaled(pred_mean - 1.96 * pred_std)[:, i],
                    from_scaled(pred_mean + 1.96 * pred_std)[:, i],
                    alpha=0.15, color="red", label="95% CI")
    ax.set_title(col[:40], fontsize=8, fontweight="bold")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3); ax.tick_params(labelsize=7)
plt.tight_layout()
plt.savefig(PLOT_DIR / "04_world_model_test_predictions.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 04_world_model_test_predictions.png saved.")


SECTION 2 — WORLD MODEL TRAINING
[WorldModel] Training 5-member ensemble, 500 epochs...


WorldModel:   0%|                                       | 0/500 [00:00<?, ?it/s]

[WorldModel] Best val NLL per member: ['-0.5801', '-0.8065', '-0.5896', '-0.8113', '-0.6837']
[WorldModel] Median MAPE on test: 1.12%
[WorldModel] Median R²   on test: 0.7203
[Plot] 03_world_model_training_curves.png saved.
[Plot] 04_world_model_test_predictions.png saved.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — MULTI-AGENT ENVIRONMENT
# ─────────────────────────────────────────────────────────────────────────────
# KurskDemoEnv implements a custom multi-agent Gymnasium-compatible environment.
# State  s_t ∈ ℝ^{STATE_DIM} — normalised demographic vector
# Action a_t^i ∈ [-1, 1]^{d_i} — continuous policy actions per agent
# Transition: s_{t+1} ~ f_θ(s_t, cat(a_t)) + ε   (ensemble world model)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 3 — MULTI-AGENT ENVIRONMENT")
print("="*70)

# ── Agent definitions ─────────────────────────────────────────────────────────

# Each dict defines one agent with its action space dimensionality and semantics.
AGENT_SPECS: List[Dict[str, Any]] = [
    # ── INSTITUTIONAL AGENTS ────────────────────────────────────────────────
    {
        "id": 0, "name": "RegionalGov",
        "type": "institutional",
        "action_dim": 5,
        "action_names": [
            "matcap_size_delta",          # Δ regional maternity capital (normalised)
            "housing_program_intensity",  # housing support for young/large families
            "benefit_access_simplicity",  # digital / one-window procedures
            "employment_women_support",   # programs for mothers with pre-school children
            "info_campaign_intensity",    # family-values communication campaigns
        ],
        "obs_groups": ["population_core", "fertility", "migration", "housing"],
    },
    {
        "id": 1, "name": "HealthcareSystem",
        "type": "institutional",
        "action_dim": 4,
        "action_names": [
            "pre_abortion_consult_intensity",  # coverage & quality of pre-abortion counselling
            "ivf_cycles_expansion",            # number of IVF cycles (Δ from baseline)
            "ob_gyn_staffing_effort",          # recruitment & retention of ob-gyns, neonatologists
            "reproductive_health_screening",   # women 15-38 screening coverage
        ],
        "obs_groups": ["reproductive_health", "healthcare_infra", "fertility"],
    },
    {
        "id": 2, "name": "EducationInfra",
        "type": "institutional",
        "action_dim": 4,
        "action_names": [
            "preschool_places_expansion",  # Δ places per 1000 children
            "after_school_group_expansion",# ГПД expansion
            "student_parent_support",      # student families measures
            "preschool_schedule_flex",     # extended / flexible childcare hours
        ],
        "obs_groups": ["education_childcare", "fertility", "population_core"],
    },
    {
        "id": 3, "name": "MigrationPolicy",
        "type": "institutional",
        "action_dim": 3,
        "action_names": [
            "labour_migration_attraction",  # investment in jobs / labour migration programmes
            "resettlement_support",         # support for resettlers (housing, payments)
            "retention_incentives",         # incentives to stay in region (youth, specialists)
        ],
        "obs_groups": ["migration", "population_core"],
    },
    # ── INDIVIDUAL / BEHAVIOURAL AGENTS ─────────────────────────────────────
    {
        "id": 4, "name": "Women",
        "type": "individual",
        "action_dim": 4,
        "action_names": [
            "birth_intention_1st",     # probability adjustment — 1st birth
            "birth_intention_2nd",     # probability adjustment — 2nd birth
            "birth_intention_3plus",   # probability adjustment — 3rd+ birth
            "abortion_avoidance",      # responsiveness to counselling
        ],
        "obs_groups": ["fertility", "reproductive_health", "education_childcare",
                       "housing"],
    },
    {
        "id": 5, "name": "Households",
        "type": "individual",
        "action_dim": 3,
        "action_names": [
            "housing_improvement_uptake",  # take-up of housing programmes
            "childcare_utilisation",       # use of childcare / preschool places
            "benefit_claim_propensity",    # probability of claiming available benefits
        ],
        "obs_groups": ["housing", "education_childcare", "fertility"],
    },
    {
        "id": 6, "name": "Families",
        "type": "individual",
        "action_dim": 3,
        "action_names": [
            "marriage_propensity",      # propensity to register marriage
            "divorce_avoidance",        # reduction in divorce rate
            "multichildren_aspiration", # aspiration for 3+ children
        ],
        "obs_groups": ["fertility", "population_core", "housing"],
    },
    {
        "id": 7, "name": "Migrants",
        "type": "individual",
        "action_dim": 3,
        "action_names": [
            "in_migration_propensity",   # propensity to migrate TO Kursk
            "out_migration_avoidance",   # propensity to STAY (reduce outflow)
            "labour_participation",      # labour market integration intensity
        ],
        "obs_groups": ["migration", "population_core"],
    },
]

# Derived dimensions
N_AGENTS     = len(AGENT_SPECS)
ACTION_DIMS  = [s["action_dim"] for s in AGENT_SPECS]
ACTION_DIM_T = sum(ACTION_DIMS)   # total joint action dim fed into world model

# Rebuild world model with correct total action dim
print(f"[Env] N_agents={N_AGENTS}, total action_dim={ACTION_DIM_T}, state_dim={STATE_DIM}")

# Reload / retrain world models with correct action dimension if needed
if ACTION_DIM_T != ACTION_DIM_WORLD:
    print(f"[Env] Rebuilding world model ensemble with action_dim={ACTION_DIM_T}...")
    world_ensemble = [
        TransitionNet(STATE_DIM, ACTION_DIM_T).to(DEVICE)
        for _ in range(ENSEMBLE_SIZE)
    ]
    world_optimizers = [
        optim.AdamW(m.parameters(), lr=5e-4, weight_decay=1e-4)
        for m in world_ensemble
    ]
    best_val_loss = [float("inf")] * ENSEMBLE_SIZE

    S_tr2, A_tr2, S1_tr2 = build_transition_dataset(HIST_SC[:len(TRAIN_SC)], ACTION_DIM_T)
    S_va2, A_va2, S1_va2 = build_transition_dataset(HIST_SC[len(TRAIN_SC):len(TRAIN_SC)+len(VAL_SC)], ACTION_DIM_T)

    for epoch in trange(WORLD_EPOCHS, desc="WorldModel (final)", ncols=80):
        for idx, (model, opt) in enumerate(zip(world_ensemble, world_optimizers)):
            model.train()
            mu, lv = model(S_tr2, A_tr2)
            delta  = S1_tr2 - S_tr2
            nll    = 0.5 * (lv + (delta - mu).pow(2) / lv.exp()).mean()
            opt.zero_grad(); nll.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            model.eval()
            with torch.no_grad():
                mv, lv_v = model(S_va2, A_va2)
                dv  = S1_va2 - S_va2
                nlv = 0.5 * (lv_v + (dv - mv).pow(2) / lv_v.exp()).mean()
                if nlv.item() < best_val_loss[idx]:
                    best_val_loss[idx] = nlv.item()
                    torch.save(model.state_dict(), MODEL_DIR / f"world_model_final_{idx}.pt")

    for idx, model in enumerate(world_ensemble):
        model.load_state_dict(
            torch.load(MODEL_DIR / f"world_model_final_{idx}.pt", map_location=DEVICE))
        model.eval()
    print(f"[Env] World model retrained. Best val NLL: {[f'{v:.4f}' for v in best_val_loss]}")

# ── Build per-agent observation indices ──────────────────────────────────────

def _obs_indices(groups: List[str]) -> List[int]:
    idx = []
    for grp in groups:
        for col in FEATURE_GROUPS.get(grp, []):
            if col in STATE_COLS:
                i = STATE_COLS.index(col)
                if i not in idx:
                    idx.append(i)
    return sorted(idx)

for spec in AGENT_SPECS:
    spec["obs_indices"] = _obs_indices(spec["obs_groups"])
    spec["obs_dim"]     = len(spec["obs_indices"])
    print(f"  {spec['name']:20s} obs_dim={spec['obs_dim']:3d}  action_dim={spec['action_dim']}")

# ── 3.1 Reward function ───────────────────────────────────────────────────────

def compute_reward(
    s_prev: np.ndarray,
    s_next: np.ndarray,
    joint_action: np.ndarray,
    agent_id: int,
) -> float:
    """
    Composite reward function for agent i:

        R_t^i = Σ_k α_k^i · Δmetric_k  −  λ · ||a_t^i||²

    where Δmetric_k = (metric_k(s_{t+1}) − metric_k(s_t)) / |metric_k(s_t)| (relative change)

    Shared global metrics are weighted by type of agent:
    ─ Institutional agents optimise population, fertility, net-migration + cost penalty
    ─ Individual agents optimise personal utility proxies (fertility, migration propensity)

    Returns scalar reward.
    """
    # Map columns to indices
    def _get(arr, name):
        if name in STATE_COLS:
            i = STATE_COLS.index(name)
            return arr[i]
        return 0.0

    def _rel_delta(s_n, s_p, name, clip=5.0):
        """Relative change, clipped to prevent extreme values."""
        v_p = _get(s_p, name)
        v_n = _get(s_n, name)
        if abs(v_p) < 1e-8:
            return 0.0
        return float(np.clip((v_n - v_p) / abs(v_p), -clip, clip))

    w = CFG["reward_w"]
    spec = AGENT_SPECS[agent_id]
    action_i = joint_action[
        sum(ACTION_DIMS[:agent_id]): sum(ACTION_DIMS[:agent_id+1])
    ]

    # ── Global metrics (scaled to [-1, +1] range approximately)
    Δpop  = _rel_delta(s_next, s_prev, "Численность населения всего")
    ΔTFR  = _rel_delta(s_next, s_prev, "СКР (всего)")
    Δmig  = _rel_delta(s_next, s_prev, "Коэффициент миграционного прироста (на 10000)")
    Δnat  = _rel_delta(s_next, s_prev, "Коэффициент естественного прироста (на 1000)")

    # Infrastructure-specific metrics
    Δhous = _rel_delta(s_next, s_prev, "Общая площадь жилья на 1 жителя (кв. м)")
    Δcare = _rel_delta(s_next, s_prev, "Укомплектованность акушерами-гинекологами (%)")

    # Action cost penalty (L2 norm) — prevents "maximum action always wins"
    cost = float(np.sum(action_i ** 2)) * abs(w["cost_penalty"])

    if spec["type"] == "institutional":
        reward = (
            w["pop_growth"]   * Δpop  +
            w["tfr"]          * ΔTFR  +
            w["net_migration"]* Δmig  +
            w["nat_increase"] * Δnat  +
            w["housing"]      * Δhous +
            w["healthcare"]   * Δcare -
            cost
        )
    else:
        # Individual agents weight their own utility more heavily
        if agent_id == 4:   # Women — fertility-focused
            reward = 0.4 * ΔTFR + 0.2 * Δnat + 0.2 * Δcare - cost
        elif agent_id == 5: # Households — housing + childcare
            reward = 0.4 * Δhous + 0.2 * ΔTFR + 0.1 * Δpop - cost
        elif agent_id == 6: # Families — fertility + stability
            reward = 0.3 * ΔTFR + 0.3 * Δnat + 0.2 * Δpop - cost
        else:               # Migrants — migration-focused
            reward = 0.5 * Δmig + 0.3 * Δpop + 0.1 * Δnat - cost

    return float(reward)


# ── 3.2 KurskDemoEnv ─────────────────────────────────────────────────────────

class KurskDemoEnv:
    """
    Custom multi-agent environment for Kursk demographic policy simulation.

    Observation:  s_t^i ∈ ℝ^{obs_dim_i}  (agent-specific view of global state)
    Action:       a_t^i ∈ [-1, 1]^{action_dim_i}
    Transition:   s_{t+1} = ensemble_world_model(s_t, cat(a_t^0,...,a_t^{N-1}))
                            + ε  (ε ~ N(0, shock_std))
    Rewards:      r_t^i = compute_reward(s_t, s_{t+1}, a_t, i)
    Terminal:     t == horizon

    Attributes
    ----------
    state       : np.ndarray of shape (STATE_DIM,) — current normalised state
    year        : int — current simulation year
    step_count  : int
    history     : dict of lists — full trajectory for logging
    """

    def __init__(
        self,
        world_models: List[TransitionNet],
        init_state: Optional[np.ndarray] = None,
        horizon: int = 25,
        shock_std: float = 0.02,
        stochastic: bool = True,
        seed: int = SEED,
    ):
        self.world_models  = world_models
        self.horizon       = horizon
        self.shock_std     = shock_std
        self.stochastic    = stochastic
        self.rng           = np.random.default_rng(seed)
        self._init_state   = init_state if init_state is not None else HIST_SC[-1].copy()
        self.reset()

    def reset(self) -> List[np.ndarray]:
        """Reset environment to initial state (year 2025)."""
        self.state      = self._init_state.copy().astype(np.float32)
        self.year       = CFG["hist_end"]
        self.step_count = 0
        self.history: Dict[str, list] = {
            "year":    [],
            "state":   [],
            "actions": [],
            "rewards": [],
        }
        return self._get_observations()

    def _get_observations(self) -> List[np.ndarray]:
        """Return agent-specific observations from global state."""
        return [
            self.state[spec["obs_indices"]].astype(np.float32)
            for spec in AGENT_SPECS
        ]

    def step(self, actions: List[np.ndarray]) -> Tuple:
        """
        Execute one step (1 year) in the environment.

        Parameters
        ----------
        actions : list of np.ndarray, one per agent

        Returns
        -------
        observations : List[np.ndarray]
        rewards      : List[float]
        done         : bool
        info         : dict
        """
        # Concatenate joint action
        joint_action = np.concatenate(actions, axis=0).astype(np.float32)
        joint_action = np.clip(joint_action, -1.0, 1.0)

        # Transition via ensemble world model (mean prediction)
        s_t = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        a_t = torch.tensor(joint_action, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            preds = []
            for m in self.world_models:
                preds.append(m.predict(s_t, a_t, deterministic=not self.stochastic))
            s_next = torch.stack(preds).mean(0).squeeze(0).cpu().numpy()

        # Optional stochastic shock (exogenous, e.g. macro/geo shocks)
        if self.stochastic:
            s_next += self.rng.normal(0, self.shock_std, size=s_next.shape).astype(np.float32)

        # Compute per-agent rewards
        rewards = [
            compute_reward(self.state, s_next, joint_action, i)
            for i in range(N_AGENTS)
        ]

        # Log trajectory
        self.history["year"].append(self.year)
        self.history["state"].append(self.state.copy())
        self.history["actions"].append(joint_action.copy())
        self.history["rewards"].append(np.array(rewards))

        self.state       = s_next
        self.year       += 1
        self.step_count += 1
        done             = self.step_count >= self.horizon

        obs  = self._get_observations()
        info = {
            "year":         self.year,
            "state_orig":   from_scaled(s_next.reshape(1, -1))[0],
            "joint_action": joint_action,
        }
        return obs, rewards, done, info

    def get_trajectory_df(self) -> pd.DataFrame:
        """Convert history to a DataFrame in original (unscaled) units."""
        years  = self.history["year"]
        states = np.array(self.history["state"])
        states_orig = from_scaled(states)
        df = pd.DataFrame(states_orig, columns=STATE_COLS, index=years)
        df.index.name = "year"
        return df

print("[Env] KurskDemoEnv defined.")

# Sanity check — random rollout
_env_test = KurskDemoEnv(world_ensemble)
_obs      = _env_test.reset()
_rand_act = [np.random.uniform(-1, 1, s["action_dim"]) for s in AGENT_SPECS]
_obs2, _rew, _done, _info = _env_test.step(_rand_act)
print(f"[Env] Random step OK — rewards: {[f'{r:.4f}' for r in _rew]}")
del _env_test, _obs, _rand_act, _obs2, _rew, _done, _info



SECTION 3 — MULTI-AGENT ENVIRONMENT
[Env] N_agents=8, total action_dim=29, state_dim=53
[Env] Rebuilding world model ensemble with action_dim=29...


WorldModel (final):   0%|                               | 0/500 [00:00<?, ?it/s]

[Env] World model retrained. Best val NLL: ['-0.5242', '-0.6194', '-0.4378', '-0.6199', '-0.4273']
  RegionalGov          obs_dim= 33  action_dim=5
  HealthcareSystem     obs_dim= 26  action_dim=4
  EducationInfra       obs_dim= 24  action_dim=4
  MigrationPolicy      obs_dim= 13  action_dim=3
  Women                obs_dim= 33  action_dim=4
  Households           obs_dim= 25  action_dim=3
  Families             obs_dim= 28  action_dim=3
  Migrants             obs_dim= 13  action_dim=3
[Env] KurskDemoEnv defined.
[Env] Random step OK — rewards: ['0.0701', '0.0587', '0.0294', '0.0934', '0.0989', '-0.0323', '0.0713', '0.3010']


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — MARL AGENTS (MADDPG + MATD3 + MAPPO baselines)
# ─────────────────────────────────────────────────────────────────────────────
# MADDPG (Lowe et al. 2017):  Centralised critic, decentralised actors.
#   π_i(o_i)  →  a_i ∈ [-1, 1]^{d_i}
#   Q_i(s, a_1,...,a_N)  →  scalar
#
# MATD3: TD3-style double critics + policy delay + target smoothing
#        (reduces overestimation bias)
#
# MAPPO: on-policy PPO with centralised value function
#        (strong baseline for cooperative tasks)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 4 — MARL AGENTS")
print("="*70)

# ── 4.1 Replay buffer ─────────────────────────────────────────────────────────

class ReplayBuffer:
    """
    Experience replay for MADDPG / MATD3.
    Stores (o_1,...,o_N, a_1,...,a_N, r_1,...,r_N, o'_1,...,o'_N, done).
    """
    def __init__(self, capacity: int, n_agents: int,
                 obs_dims: List[int], act_dims: List[int]):
        self.capacity  = capacity
        self.n_agents  = n_agents
        self.obs_dims  = obs_dims
        self.act_dims  = act_dims
        self._ptr      = 0
        self._size     = 0

        self.obs  = [np.zeros((capacity, d), dtype=np.float32) for d in obs_dims]
        self.nobs = [np.zeros((capacity, d), dtype=np.float32) for d in obs_dims]
        self.acts = [np.zeros((capacity, d), dtype=np.float32) for d in act_dims]
        self.rews = np.zeros((capacity, n_agents), dtype=np.float32)
        self.done = np.zeros((capacity, 1),         dtype=np.float32)

    def push(self, obs, acts, rews, nobs, done):
        i = self._ptr
        for j in range(self.n_agents):
            self.obs[j][i]  = obs[j]
            self.nobs[j][i] = nobs[j]
            self.acts[j][i] = acts[j]
        self.rews[i] = rews
        self.done[i] = float(done)
        self._ptr  = (self._ptr + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self._size, size=batch_size)
        def _t(arr):
            return torch.tensor(arr[idx], dtype=torch.float32).to(DEVICE)
        return (
            [_t(self.obs[j])  for j in range(self.n_agents)],
            [_t(self.acts[j]) for j in range(self.n_agents)],
            _t(self.rews),
            [_t(self.nobs[j]) for j in range(self.n_agents)],
            _t(self.done),
        )

    def __len__(self):
        return self._size


# ── 4.2 Network architectures ─────────────────────────────────────────────────

class Actor(nn.Module):
    """
    Decentralised actor π_i(o_i) → a_i ∈ (-1, 1)^{d_i}
    Uses tanh output to bound actions.
    """
    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.net(obs))


class CentralisedCritic(nn.Module):
    """
    Centralised Q-function Q_i(s_global, a_1,...,a_N) → scalar.
    Input:  concatenation of all observations + all actions.
    Used by MADDPG and MATD3 (double critics).
    """
    def __init__(self, total_obs_dim: int, total_act_dim: int, hidden: int = 256):
        super().__init__()
        in_dim = total_obs_dim + total_act_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, all_obs: torch.Tensor, all_acts: torch.Tensor) -> torch.Tensor:
        x = torch.cat([all_obs, all_acts], dim=-1)
        return self.net(x)


# TD3-style double critic
class DoubleCritic(nn.Module):
    """Two independent critics for MATD3 (min trick)."""
    def __init__(self, total_obs_dim: int, total_act_dim: int, hidden: int = 256):
        super().__init__()
        self.Q1 = CentralisedCritic(total_obs_dim, total_act_dim, hidden)
        self.Q2 = CentralisedCritic(total_obs_dim, total_act_dim, hidden)

    def forward(self, o, a):
        return self.Q1(o, a), self.Q2(o, a)

    def Q_min(self, o, a):
        q1, q2 = self.forward(o, a)
        return torch.min(q1, q2)


# ── 4.3 MADDPG agent ──────────────────────────────────────────────────────────

class MADDPGAgent:
    """
    Single agent in the MADDPG framework.
    Maintains:  actor, target_actor, critic, target_critic
    Updates via:
        critic_loss = MSE( r + γ Q_target(o', a'_target) , Q(o, a) )
        actor_loss  = -E[ Q(o, π(o)) ]
    Soft updates: θ_target ← τ·θ + (1-τ)·θ_target
    """
    def __init__(self, agent_id: int, total_obs_dim: int, total_act_dim: int):
        spec        = AGENT_SPECS[agent_id]
        obs_dim     = spec["obs_dim"]
        action_dim  = spec["action_dim"]
        hidden      = CFG["hidden_dim"]
        lr_a        = CFG["actor_lr"]
        lr_c        = CFG["critic_lr"]
        self.id     = agent_id
        self.γ      = CFG["gamma"]
        self.τ      = CFG["tau"]
        self.name   = spec["name"]

        self.actor         = Actor(obs_dim, action_dim, hidden).to(DEVICE)
        self.actor_target  = deepcopy(self.actor).to(DEVICE)
        self.critic        = CentralisedCritic(total_obs_dim, total_act_dim, hidden).to(DEVICE)
        self.critic_target = deepcopy(self.critic).to(DEVICE)

        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=lr_a)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr_c)

        # Noise (Ornstein-Uhlenbeck for exploration)
        self.action_dim  = action_dim
        self.noise_scale = 0.2
        self._noise_state = np.zeros(action_dim)

    def act(self, obs: np.ndarray, explore: bool = True) -> np.ndarray:
        """Select action given local observation."""
        with torch.no_grad():
            o = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            a = self.actor(o).squeeze(0).cpu().numpy()
        if explore:
            # OU noise
            theta, sigma, dt = 0.15, 0.2, 1.0
            self._noise_state += (
                -theta * self._noise_state * dt
                + sigma * np.sqrt(dt) * np.random.randn(self.action_dim)
            )
            a = np.clip(a + self._noise_state * self.noise_scale, -1.0, 1.0)
        return a.astype(np.float32)

    def update(
        self,
        batch: Tuple,
        all_actors: List["MADDPGAgent"],
        all_actor_targets: List["MADDPGAgent"],
    ) -> Tuple[float, float]:
        """
        One gradient step for critic and actor.
        Returns (critic_loss, actor_loss) as Python floats.
        """
        obs_list, act_list, rew_batch, nobs_list, done_batch = batch
        bs = rew_batch.shape[0]

        # ── Global state/action tensors
        all_obs  = torch.cat(obs_list,  dim=-1)   # (B, sum_obs_dims)
        all_acts = torch.cat(act_list,  dim=-1)   # (B, sum_act_dims)
        all_nobs = torch.cat(nobs_list, dim=-1)

        # ── Target actions from target actors
        with torch.no_grad():
            target_acts = torch.cat([
                ag.actor_target(nobs_list[j])
                for j, ag in enumerate(all_actors)
            ], dim=-1)
            y = (
                rew_batch[:, self.id].unsqueeze(1)
                + self.γ * (1 - done_batch) * self.critic_target(all_nobs, target_acts)
            )

        # ── Critic update
        q = self.critic(all_obs, all_acts)
        critic_loss = F.mse_loss(q, y.detach())
        self.critic_opt.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_opt.step()

        # ── Actor update (only update own action, freeze others)
        curr_acts = []
        for j, ag in enumerate(all_actors):
            if j == self.id:
                curr_acts.append(ag.actor(obs_list[j]))      # differentiable
            else:
                curr_acts.append(act_list[j].detach())       # stop gradient
        actor_loss = -self.critic(all_obs, torch.cat(curr_acts, dim=-1)).mean()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
        self.actor_opt.step()

        # ── Soft target updates
        for p, pt in zip(self.critic.parameters(), self.critic_target.parameters()):
            pt.data.copy_(self.τ * p.data + (1 - self.τ) * pt.data)
        for p, pt in zip(self.actor.parameters(), self.actor_target.parameters()):
            pt.data.copy_(self.τ * p.data + (1 - self.τ) * pt.data)

        return critic_loss.item(), actor_loss.item()


# ── 4.4 MATD3 agent (variant) ─────────────────────────────────────────────────

class MATD3Agent(MADDPGAgent):
    """
    MATD3: MADDPG + double critics + policy delay + target action smoothing.
    Inherits from MADDPGAgent, overrides update().
    """
    def __init__(self, agent_id: int, total_obs_dim: int, total_act_dim: int,
                 policy_delay: int = 2):
        super().__init__(agent_id, total_obs_dim, total_act_dim)
        # Replace single critic with double critic
        hidden = CFG["hidden_dim"]
        self.critic        = DoubleCritic(total_obs_dim, total_act_dim, hidden).to(DEVICE)
        self.critic_target = deepcopy(self.critic).to(DEVICE)
        self.critic_opt    = optim.Adam(self.critic.parameters(), lr=CFG["critic_lr"])
        self.policy_delay  = policy_delay
        self._update_step  = 0
        self.target_noise  = 0.2
        self.noise_clip    = 0.5

    def update(self, batch, all_actors, all_actor_targets) -> Tuple[float, float]:
        self._update_step += 1
        obs_list, act_list, rew_batch, nobs_list, done_batch = batch
        all_obs  = torch.cat(obs_list,  dim=-1)
        all_acts = torch.cat(act_list,  dim=-1)
        all_nobs = torch.cat(nobs_list, dim=-1)

        with torch.no_grad():
            target_acts_list = []
            for j, ag in enumerate(all_actors):
                ta = ag.actor_target(nobs_list[j])
                noise = (torch.randn_like(ta) * self.target_noise).clamp(
                    -self.noise_clip, self.noise_clip)
                ta = (ta + noise).clamp(-1.0, 1.0)
                target_acts_list.append(ta)
            tgt_acts = torch.cat(target_acts_list, dim=-1)
            q_min = self.critic.Q_min(all_nobs, tgt_acts)
            y = rew_batch[:, self.id].unsqueeze(1) + self.γ * (1 - done_batch) * q_min

        q1, q2 = self.critic(all_obs, all_acts)
        critic_loss = F.mse_loss(q1, y.detach()) + F.mse_loss(q2, y.detach())
        self.critic_opt.zero_grad(); critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_opt.step()

        actor_loss = torch.tensor(0.0)
        if self._update_step % self.policy_delay == 0:
            curr = [
                all_actors[j].actor(obs_list[j]) if j == self.id else act_list[j].detach()
                for j in range(N_AGENTS)
            ]
            actor_loss = -self.critic.Q1(all_obs, torch.cat(curr, dim=-1)).mean()
            self.actor_opt.zero_grad(); actor_loss.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            self.actor_opt.step()
            for p, pt in zip(self.actor.parameters(), self.actor_target.parameters()):
                pt.data.copy_(self.τ * p.data + (1 - self.τ) * pt.data)

        for p, pt in zip(self.critic.parameters(), self.critic_target.parameters()):
            pt.data.copy_(self.τ * p.data + (1 - self.τ) * pt.data)

        return critic_loss.item(), actor_loss.item()


# ── 4.5 MAPPO (on-policy baseline) ────────────────────────────────────────────

class MAPPOActor(nn.Module):
    """PPO actor: outputs action mean (tanh-squashed) and log_std."""
    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu_head     = nn.Linear(hidden, action_dim)
        self.log_std     = nn.Parameter(torch.zeros(action_dim))

    def forward(self, obs):
        h   = self.net(obs)
        mu  = torch.tanh(self.mu_head(h))
        std = self.log_std.exp().expand_as(mu)
        return torch.distributions.Normal(mu, std)

    def act(self, obs: np.ndarray, deterministic=False) -> Tuple[np.ndarray, float]:
        with torch.no_grad():
            o    = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            dist = self.forward(o)
            a    = dist.mean if deterministic else dist.sample()
            logp = dist.log_prob(a).sum(-1).item()
        return a.squeeze(0).cpu().numpy(), logp


class MAPPOCritic(nn.Module):
    """Centralised value function V(global_state)."""
    def __init__(self, total_obs_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(total_obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),         nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, global_obs: torch.Tensor) -> torch.Tensor:
        return self.net(global_obs)


print("[Agents] All agent classes defined (MADDPG, MATD3, MAPPO).")


SECTION 4 — MARL AGENTS
[Agents] All agent classes defined (MADDPG, MATD3, MAPPO).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 5 — TRAINING")
print("="*70)

# ── Pre-compute global dims ───────────────────────────────────────────────────
OBS_DIMS      = [s["obs_dim"]    for s in AGENT_SPECS]
TOTAL_OBS_DIM = sum(OBS_DIMS)
TOTAL_ACT_DIM = sum(ACTION_DIMS)

print(f"[Train] total_obs_dim={TOTAL_OBS_DIM}, total_act_dim={TOTAL_ACT_DIM}")

def run_maddpg_training(
    algo_class,
    algo_name: str,
    n_episodes: int,
    warmup_steps: int,
) -> Tuple[List[MADDPGAgent], Dict]:
    """
    Full MADDPG / MATD3 training loop.

    Parameters
    ----------
    algo_class   : MADDPGAgent or MATD3Agent
    algo_name    : label for logging
    n_episodes   : total training episodes
    warmup_steps : random exploration before gradient updates

    Returns
    -------
    agents       : trained agent list
    logs         : dict of training statistics
    """
    agents = [
        algo_class(i, TOTAL_OBS_DIM, TOTAL_ACT_DIM)
        for i in range(N_AGENTS)
    ]
    buffer = ReplayBuffer(
        CFG["buffer_size"], N_AGENTS, OBS_DIMS, ACTION_DIMS
    )
    env = KurskDemoEnv(
        world_ensemble,
        horizon=CFG["max_steps"],
        shock_std=CFG["shock_std"],
        stochastic=True,
    )

    logs: Dict[str, list] = {
        "ep_return":     [],   # sum of per-agent mean reward per episode
        "critic_losses": defaultdict(list),
        "actor_losses":  defaultdict(list),
        "ep_length":     [],
    }

    global_step = 0

    for ep in trange(n_episodes, desc=f"{algo_name} Training", ncols=80):
        obs  = env.reset()
        ep_return   = 0.0
        ep_rew_list = []

        for t in range(CFG["max_steps"]):
            # Action selection
            if global_step < warmup_steps:
                actions = [
                    np.random.uniform(-1, 1, AGENT_SPECS[i]["action_dim"]).astype(np.float32)
                    for i in range(N_AGENTS)
                ]
            else:
                actions = [ag.act(obs[i], explore=True) for i, ag in enumerate(agents)]

            nobs, rewards, done, info = env.step(actions)
            buffer.push(obs, actions, rewards, nobs, float(done))

            ep_return += float(np.mean(rewards))
            ep_rew_list.append(rewards)
            obs          = nobs
            global_step += 1

            # Gradient update
            if len(buffer) > CFG["batch_size"] and global_step > warmup_steps:
                batch = buffer.sample(CFG["batch_size"])
                for ag in agents:
                    cl, al = ag.update(batch, agents, agents)
                    logs["critic_losses"][ag.name].append(cl)
                    logs["actor_losses"][ag.name].append(al)

            if done:
                break

        logs["ep_return"].append(ep_return)
        logs["ep_length"].append(env.step_count)

        # Decay exploration noise
        for ag in agents:
            ag.noise_scale = max(0.01, ag.noise_scale * 0.9995)

        # Checkpoint every 500 episodes
        if (ep + 1) % 500 == 0 or ep == n_episodes - 1:
            for ag in agents:
                torch.save(ag.actor.state_dict(),
                           MODEL_DIR / f"{algo_name}_{ag.name}_actor_ep{ep+1}.pt")
            print(f"\n  [{algo_name}] Ep {ep+1}/{n_episodes} | "
                  f"mean_return={np.mean(logs['ep_return'][-50:]):.4f} | "
                  f"buffer={len(buffer)}")

    return agents, logs


# ── Train MADDPG ──────────────────────────────────────────────────────────────
print("\n[Train] Starting MADDPG...")
t0 = time.time()
maddpg_agents, maddpg_logs = run_maddpg_training(
    MADDPGAgent, "MADDPG",
    n_episodes=CFG["n_episodes"],
    warmup_steps=CFG["warmup_steps"],
)
print(f"[Train] MADDPG done in {(time.time()-t0)/60:.1f} min")

# ── Train MATD3 ───────────────────────────────────────────────────────────────
print("\n[Train] Starting MATD3...")
t0 = time.time()
matd3_agents, matd3_logs = run_maddpg_training(
    MATD3Agent, "MATD3",
    n_episodes=CFG["n_episodes"],
    warmup_steps=CFG["warmup_steps"],
)
print(f"[Train] MATD3 done in {(time.time()-t0)/60:.1f} min")

# ── Save training logs ────────────────────────────────────────────────────────
for name, logs in [("MADDPG", maddpg_logs), ("MATD3", matd3_logs)]:
    pd.DataFrame({"episode": range(len(logs["ep_return"])),
                  "ep_return": logs["ep_return"],
                  "ep_length": logs["ep_length"]}).to_csv(
        REPORT_DIR / f"{name}_training_log.csv", index=False)

# ── Training curves plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Кривые обучения MADDPG и MATD3", fontsize=13, fontweight="bold")

def smooth(x, w=50):
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w)/w, mode="valid")

for ax, (name, logs) in zip(axes, [("MADDPG", maddpg_logs), ("MATD3", matd3_logs)]):
    raw  = np.array(logs["ep_return"])
    sraw = smooth(raw, 50)
    ax.plot(raw, alpha=0.2, color="steelblue", linewidth=0.5)
    ax.plot(np.arange(len(sraw)) + 25, sraw, color="steelblue", linewidth=2, label="MA-50")
    ax.set_title(f"{name} — Суммарная награда", fontweight="bold")
    ax.set_xlabel("Эпизод"); ax.set_ylabel("Средняя награда агентов")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "05_training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 05_training_curves.png saved.")


SECTION 5 — TRAINING
[Train] total_obs_dim=195, total_act_dim=29

[Train] Starting MADDPG...


MADDPG Training:   0%|                                  | 0/600 [00:00<?, ?it/s]


  [MADDPG] Ep 500/600 | mean_return=-1.5071 | buffer=12500

  [MADDPG] Ep 600/600 | mean_return=-1.6154 | buffer=15000
[Train] MADDPG done in 18.3 min

[Train] Starting MATD3...


MATD3 Training:   0%|                                   | 0/600 [00:00<?, ?it/s]


  [MATD3] Ep 500/600 | mean_return=-1.7265 | buffer=12500

  [MATD3] Ep 600/600 | mean_return=-1.9407 | buffer=15000
[Train] MATD3 done in 19.9 min
[Plot] 05_training_curves.png saved.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — EVALUATION: MARL vs BASELINE TRAJECTORIES
# ─────────────────────────────────────────────────────────────────────────────
# Baseline = "business-as-usual": zero-action rollout (no policy interventions)
# MARL trajectory = greedy rollout with trained MADDPG / MATD3 agents
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 6 — EVALUATION")
print("="*70)

def rollout(
    agents_or_none,
    n_scenarios: int = 1,
    stochastic: bool = True,
    horizon: int = CFG["horizon"],
    deterministic_policy: bool = True,
) -> pd.DataFrame:
    """
    Run `n_scenarios` rollouts and return the MEAN trajectory as a DataFrame.

    agents_or_none : list of trained agents, or None (→ baseline: zero actions)
    """
    all_trajs = []
    for sc in range(n_scenarios):
        env  = KurskDemoEnv(world_ensemble, horizon=horizon,
                            shock_std=CFG["shock_std"] if stochastic else 0.0,
                            stochastic=stochastic, seed=SEED + sc)
        obs  = env.reset()
        done = False
        while not done:
            if agents_or_none is None:
                actions = [np.zeros(AGENT_SPECS[i]["action_dim"], dtype=np.float32)
                           for i in range(N_AGENTS)]
            else:
                actions = [ag.act(obs[i], explore=not deterministic_policy)
                           for i, ag in enumerate(agents_or_none)]
            obs, _, done, _ = env.step(actions)
        all_trajs.append(env.get_trajectory_df())

    # Aggregate: mean across scenarios
    combined = pd.concat(all_trajs).groupby(level=0).mean()
    return combined


print("[Eval] Running baseline rollout (zero actions, 30 scenarios)...")
baseline_traj = rollout(None, n_scenarios=CFG["n_scenarios"], stochastic=True)

print("[Eval] Running MADDPG rollout...")
maddpg_traj = rollout(maddpg_agents, n_scenarios=CFG["n_scenarios"], stochastic=True)

print("[Eval] Running MATD3 rollout...")
matd3_traj  = rollout(matd3_agents, n_scenarios=CFG["n_scenarios"], stochastic=True)

# ── Also run deterministic rollouts for 2036 and 2050 snapshots ──────────────
maddpg_det_2036 = rollout(maddpg_agents, n_scenarios=1, stochastic=False,
                          horizon=10, deterministic_policy=True)
maddpg_det_2050 = rollout(maddpg_agents, n_scenarios=1, stochastic=False,
                          horizon=25, deterministic_policy=True)
matd3_det_2050  = rollout(matd3_agents,  n_scenarios=1, stochastic=False,
                          horizon=25, deterministic_policy=True)

# ── Prepend historical data for continuity ────────────────────────────────────
hist_display = HIST_DF.copy()   # original-scale historical data

def prepend_hist(traj_df):
    return pd.concat([hist_display, traj_df]).sort_index()

baseline_full = prepend_hist(baseline_traj)
maddpg_full   = prepend_hist(maddpg_traj)
matd3_full    = prepend_hist(matd3_traj)

# ── Save trajectories ─────────────────────────────────────────────────────────
baseline_full.to_csv(REPORT_DIR / "trajectory_baseline.csv")
maddpg_full.to_csv(  REPORT_DIR / "trajectory_maddpg.csv")
matd3_full.to_csv(   REPORT_DIR / "trajectory_matd3.csv")
FORE_DF.to_csv(      REPORT_DIR / "trajectory_synthetic_forecast.csv")
print("[Eval] Trajectories saved.")

# ── Effect size: MARL delta vs baseline ──────────────────────────────────────
def compute_effect(marl_traj, base_traj, target_year=2050):
    """Compute % difference between MARL and baseline at a target year."""
    effects = {}
    for col in STATE_COLS:
        if col not in marl_traj.columns:
            continue
        b = base_traj.loc[target_year, col] if target_year in base_traj.index else np.nan
        m = marl_traj.loc[target_year, col] if target_year in marl_traj.index else np.nan
        if pd.isna(b) or abs(b) < 1e-6:
            effects[col] = np.nan
        else:
            effects[col] = (m - b) / abs(b) * 100.0
    return pd.Series(effects, name="delta_pct")

eff_maddpg_2036 = compute_effect(maddpg_traj, baseline_traj, 2035)
eff_maddpg_2050 = compute_effect(maddpg_traj, baseline_traj, 2049)
eff_matd3_2050  = compute_effect(matd3_traj,  baseline_traj, 2049)

effect_df = pd.DataFrame({
    "MADDPG_Δ%_2036": eff_maddpg_2036,
    "MADDPG_Δ%_2050": eff_maddpg_2050,
    "MATD3_Δ%_2050":  eff_matd3_2050,
}).dropna()
effect_df.to_csv(REPORT_DIR / "marl_vs_baseline_effect_sizes.csv")
print(f"[Eval] Top MADDPG effects (2050):\n{effect_df['MADDPG_Δ%_2050'].nlargest(5)}")


SECTION 6 — EVALUATION
[Eval] Running baseline rollout (zero actions, 30 scenarios)...
[Eval] Running MADDPG rollout...
[Eval] Running MATD3 rollout...
[Eval] Trajectories saved.
[Eval] Top MADDPG effects (2050):
Отношение числа браков к числу разводов                           23.845297
Коэффициент естественного прироста (на 1000)                      20.542810
Численность мигрантов моложе трудоспособного возраста (сальдо)    11.051595
Число абортов на 100 родов                                         9.511422
Обеспеченность койками (на 10 тыс.)                                8.336448
Name: MADDPG_Δ%_2050, dtype: float32


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — POLICY ANALYSIS & ACTION PREFERENCE MAPS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 7 — POLICY ANALYSIS")
print("="*70)

def collect_action_log(agents, n_rollouts=10, horizon=25):
    """
    Collect detailed action log: for each year, for each agent, record
    the chosen action vector.  Returns DataFrame with columns:
        year, agent_name, action_name, action_value
    """
    records = []
    for sc in range(n_rollouts):
        env  = KurskDemoEnv(world_ensemble, horizon=horizon,
                            shock_std=CFG["shock_std"], stochastic=True, seed=SEED+sc)
        obs  = env.reset()
        done = False
        t    = 0
        while not done:
            actions = [ag.act(obs[i], explore=False) for i, ag in enumerate(agents)]
            for i, ag in enumerate(agents):
                for j, aname in enumerate(AGENT_SPECS[i]["action_names"]):
                    records.append({
                        "scenario":     sc,
                        "year":         CFG["hist_end"] + t + 1,
                        "agent_id":     i,
                        "agent_name":   AGENT_SPECS[i]["name"],
                        "action_idx":   j,
                        "action_name":  aname,
                        "action_value": float(actions[i][j]),
                    })
            obs, _, done, _ = env.step(actions)
            t += 1
    return pd.DataFrame(records)

print("[Policy] Collecting MADDPG action log...")
maddpg_action_log = collect_action_log(maddpg_agents, n_rollouts=10)
print("[Policy] Collecting MATD3 action log...")
matd3_action_log  = collect_action_log(matd3_agents,  n_rollouts=10)

maddpg_action_log.to_csv(REPORT_DIR / "maddpg_action_log.csv", index=False)
matd3_action_log.to_csv( REPORT_DIR / "matd3_action_log.csv",  index=False)

# ── Action preference statistics ──────────────────────────────────────────────
action_pref_maddpg = (
    maddpg_action_log
    .groupby(["agent_name", "action_name"])["action_value"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
action_pref_matd3 = (
    matd3_action_log
    .groupby(["agent_name", "action_name"])["action_value"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
action_pref_maddpg.to_csv(REPORT_DIR / "maddpg_action_preferences.csv")
action_pref_matd3.to_csv( REPORT_DIR / "matd3_action_preferences.csv")
print("[Policy] Action preference tables saved.")

# ── Action–outcome correlation (which actions correlate with TFR improvement) ─
def action_outcome_correlation(action_log, traj_df, outcome_col):
    """
    For each (agent, action) pair compute Spearman correlation
    between mean action level per year and outcome delta per year.
    """
    # yearly mean action value
    yearly_actions = (
        action_log.groupby(["year", "agent_name", "action_name"])["action_value"].mean()
        .reset_index()
    )
    # yearly delta of outcome
    if outcome_col not in traj_df.columns:
        return pd.DataFrame()
    outcome = traj_df[outcome_col].diff().dropna()
    results = []
    for (agent, aname), grp in yearly_actions.groupby(["agent_name", "action_name"]):
        grp = grp.set_index("year")
        common = grp.index.intersection(outcome.index)
        if len(common) < 4:
            continue
        rho, pval = spearmanr(grp.loc[common, "action_value"], outcome.loc[common])
        results.append({
            "agent":       agent,
            "action":      aname,
            "outcome":     outcome_col,
            "spearman_r":  round(rho,  4),
            "p_value":     round(pval, 4),
        })
    return pd.DataFrame(results).sort_values("spearman_r", ascending=False)

corr_tfr  = action_outcome_correlation(maddpg_action_log, maddpg_traj, "СКР (всего)")
corr_mig  = action_outcome_correlation(maddpg_action_log, maddpg_traj,
            "Коэффициент миграционного прироста (на 10000)")
corr_pop  = action_outcome_correlation(maddpg_action_log, maddpg_traj,
            "Численность населения всего")

corr_tfr.to_csv( REPORT_DIR / "action_outcome_corr_tfr.csv",  index=False)
corr_mig.to_csv( REPORT_DIR / "action_outcome_corr_mig.csv",  index=False)
corr_pop.to_csv( REPORT_DIR / "action_outcome_corr_pop.csv",  index=False)
print("[Policy] Action–outcome correlations saved.")


SECTION 7 — POLICY ANALYSIS
[Policy] Collecting MADDPG action log...
[Policy] Collecting MATD3 action log...
[Policy] Action preference tables saved.
[Policy] Action–outcome correlations saved.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — PUBLICATION-GRADE VISUALISATIONS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 8 — VISUALISATIONS")
print("="*70)

HIST_COLOR     = "#2c7bb6"
BASELINE_COLOR = "#d7191c"
MADDPG_COLOR   = "#1a9641"
MATD3_COLOR    = "#f4a742"
FORE_COLOR     = "#7b2d8b"
ALPHA_BAND     = 0.15

PLOT_VARS = [
    ("Численность населения всего",                       "Численность населения, чел.", "Тыс. чел."),
    ("СКР (всего)",                                       "СКР (суммарный к-т рождаемости)", ""),
    ("Коэффициент естественного прироста (на 1000)",      "Коэффициент естественного прироста", "на 1000"),
    ("Коэффициент миграционного прироста (на 10000)",     "Коэффициент миграционного прироста", "на 10000"),
    ("Число абортов на 100 родов",                        "Аборты на 100 родов", ""),
    ("Число циклов ЭКО",                                  "Число циклов ЭКО", ""),
    ("Общая площадь жилья на 1 жителя (кв. м)",          "Жильё на 1 жителя, кв. м", "кв. м"),
    ("Обеспеченность местами в ДОУ (на 1000 детей)",     "Обеспеченность ДОУ (на 1000 детей)", ""),
    ("Укомплектованность акушерами-гинекологами (%)",    "Укомплект. акушер-гинекол., %", "%"),
]

# ── 8.1 Main trajectory comparison ───────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle("Курская область — сравнение траекторий: Базовый сценарий vs MADDPG vs MATD3\n"
             "(1991–2050, серая зона = ±1σ стохастических роллаутов)",
             fontsize=14, fontweight="bold", y=1.02)

for ax, (col, title, unit) in zip(axes.flat, PLOT_VARS):
    if col not in HIST_DF.columns:
        ax.axis("off"); continue

    years_h  = HIST_DF.index.values
    years_f  = baseline_traj.index.values

    ax.plot(years_h, HIST_DF[col].values,
            color=HIST_COLOR, linewidth=2.5, label="История")
    ax.plot(years_f, baseline_traj[col].values,
            color=BASELINE_COLOR, linewidth=2, linestyle="--", label="Базовый")
    ax.plot(years_f, maddpg_traj[col].values,
            color=MADDPG_COLOR, linewidth=2, linestyle="-", label="MADDPG")
    ax.plot(years_f, matd3_traj[col].values,
            color=MATD3_COLOR, linewidth=2, linestyle="-.", label="MATD3")

    if col in FORE_DF.columns:
        ax.plot(FORE_DF.index, FORE_DF[col].values,
                color=FORE_COLOR, linewidth=1.5, linestyle=":", alpha=0.7, label="Синт. прогноз")

    ax.axvline(CFG["hist_end"], color="black", linewidth=1.2, linestyle="--", alpha=0.5)
    ax.axvline(2036, color="gray", linewidth=0.8, linestyle=":", alpha=0.5)
    ax.set_title(f"{title}" + (f" [{unit}]" if unit else ""),
                 fontsize=9, fontweight="bold")
    ax.legend(fontsize=6, loc="best")
    ax.grid(True, alpha=0.25)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(PLOT_DIR / "06_trajectory_comparison_main.png", dpi=180, bbox_inches="tight")
plt.close()
print("[Plot] 06_trajectory_comparison_main.png saved.")

# ── 8.2 Effect size bar chart ─────────────────────────────────────────────────
top_eff = effect_df["MADDPG_Δ%_2050"].dropna().nlargest(12)
bot_eff = effect_df["MADDPG_Δ%_2050"].dropna().nsmallest(6)
display_eff = pd.concat([top_eff, bot_eff]).sort_values()

fig, ax = plt.subplots(figsize=(12, 8))
colors_bar = [MADDPG_COLOR if v >= 0 else BASELINE_COLOR for v in display_eff.values]
bars = ax.barh(range(len(display_eff)), display_eff.values, color=colors_bar, alpha=0.85)
ax.set_yticks(range(len(display_eff)))
ax.set_yticklabels(
    [c[:55] for c in display_eff.index],
    fontsize=8
)
ax.axvline(0, color="black", linewidth=1.0)
ax.set_xlabel("Δ% (MADDPG vs Baseline) к 2050 году", fontsize=10)
ax.set_title("Размер эффекта MADDPG: отклонение от базового сценария к 2050 году",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "07_effect_size_2050.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 07_effect_size_2050.png saved.")

# ── 8.3 Action heatmap — MADDPG by year ──────────────────────────────────────
pivot_actions = (
    maddpg_action_log
    .groupby(["year", "agent_name", "action_name"])["action_value"].mean()
    .reset_index()
    .assign(label=lambda df: df["agent_name"] + " | " + df["action_name"])
    .pivot_table(index="year", columns="label", values="action_value")
)

fig, ax = plt.subplots(figsize=(22, 8))
sns.heatmap(pivot_actions.T, cmap="RdYlGn", center=0,
            linewidths=0.3, ax=ax, annot=False,
            xticklabels=True, yticklabels=True,
            cbar_kws={"label": "Интенсивность действия [-1, +1]"})
ax.set_title("Тепловая карта действий агентов MADDPG по годам (2026–2050)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Год"); ax.set_ylabel("Агент | Действие")
ax.tick_params(axis="y", labelsize=6)
ax.tick_params(axis="x", labelsize=8, rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / "08_action_heatmap_maddpg.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 08_action_heatmap_maddpg.png saved.")

# ── 8.4 Action–outcome correlation chart ─────────────────────────────────────
if not corr_tfr.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Корреляция действий агентов с ключевыми результатами (MADDPG)",
                 fontsize=12, fontweight="bold")
    for ax, (corr_df, outcome) in zip(axes, [
        (corr_tfr, "СКР"),
        (corr_mig, "Миграционный прирост"),
        (corr_pop, "Численность населения"),
    ]):
        if corr_df.empty:
            ax.axis("off"); continue
        top = corr_df.head(10)
        c   = [MADDPG_COLOR if r >= 0 else BASELINE_COLOR for r in top["spearman_r"]]
        ax.barh(top["agent"] + " | " + top["action"], top["spearman_r"],
                color=c, alpha=0.85)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"vs {outcome}", fontsize=10, fontweight="bold")
        ax.set_xlabel("ρ Спирмена", fontsize=9)
        ax.tick_params(labelsize=7)
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "09_action_outcome_correlations.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Plot] 09_action_outcome_correlations.png saved.")

# ── 8.5 Radar / spider chart — 2036 KPI comparison ───────────────────────────
RADAR_METRICS = [
    "СКР (всего)",
    "Коэффициент естественного прироста (на 1000)",
    "Коэффициент миграционного прироста (на 10000)",
    "Обеспеченность местами в ДОУ (на 1000 детей)",
    "Число абортов на 100 родов",       # lower = better → we invert
    "Общая площадь жилья на 1 жителя (кв. м)",
]

def get_kpi_at(traj, year, cols):
    """Return normalised KPI values at a given year from a trajectory."""
    if year not in traj.index:
        year = traj.index[abs(traj.index - year).argmin()]
    vals = []
    for c in cols:
        v = float(traj.loc[year, c]) if c in traj.columns else 0.0
        vals.append(v)
    return np.array(vals)

def normalise_radar(vals_list, invert_idx=[4]):
    """Min-max normalise across trajectories; invert selected indices."""
    arr = np.array(vals_list)   # (n_trajs, n_metrics)
    mn, mx = arr.min(0), arr.max(0)
    rng = mx - mn
    rng[rng == 0] = 1
    norm = (arr - mn) / rng
    for i in invert_idx:
        norm[:, i] = 1 - norm[:, i]
    return norm

yr_radar = 2035
kpi_base   = get_kpi_at(baseline_traj, yr_radar, RADAR_METRICS)
kpi_maddpg = get_kpi_at(maddpg_traj,  yr_radar, RADAR_METRICS)
kpi_matd3  = get_kpi_at(matd3_traj,   yr_radar, RADAR_METRICS)

kpi_norm = normalise_radar([kpi_base, kpi_maddpg, kpi_matd3])

labels  = [c[:30] for c in RADAR_METRICS]
N_ax    = len(labels)
angles  = np.linspace(0, 2 * np.pi, N_ax, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for vals, color, label in zip(
    kpi_norm, [BASELINE_COLOR, MADDPG_COLOR, MATD3_COLOR],
    ["Базовый", "MADDPG", "MATD3"]
):
    v = vals.tolist() + vals[:1].tolist()
    ax.plot(angles, v, color=color, linewidth=2, label=label)
    ax.fill(angles, v, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=8)
ax.set_title(f"Радарная диаграмма KPI — {yr_radar} год\n(нормализовано, ↑ лучше)",
             fontsize=12, fontweight="bold", y=1.12)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig(PLOT_DIR / "10_radar_kpi_2036.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 10_radar_kpi_2036.png saved.")

# ── 8.6 Reward decomposition per agent per episode ───────────────────────────
# Re-run short eval and collect per-agent per-step rewards
eval_env = KurskDemoEnv(world_ensemble, horizon=25, shock_std=0.0, stochastic=False)
obs      = eval_env.reset()
step_rewards_maddpg = []
done = False
while not done:
    acts = [maddpg_agents[i].act(obs[i], explore=False) for i in range(N_AGENTS)]
    obs, rew, done, _ = eval_env.step(acts)
    step_rewards_maddpg.append(rew)

rew_arr = np.array(step_rewards_maddpg)  # (T, N_agents)
years_e = list(range(CFG["hist_end"]+1, CFG["hist_end"]+1+len(rew_arr)))

fig, ax = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(rew_arr))
agent_colors = plt.cm.Set2.colors
for i, spec in enumerate(AGENT_SPECS):
    ax.bar(years_e, rew_arr[:, i], bottom=bottom,
           label=spec["name"], color=agent_colors[i % len(agent_colors)], alpha=0.85)
    bottom += rew_arr[:, i]
ax.set_xlabel("Год"); ax.set_ylabel("Вознаграждение")
ax.set_title("Разложение суммарной награды по агентам (MADDPG, детерминированный роллаут)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, ncol=2); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "11_reward_decomposition.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 11_reward_decomposition.png saved.")

# ── 8.7 World model uncertainty bands ────────────────────────────────────────
wm_col_idx = {c: STATE_COLS.index(c) for c in STATE_COLS
              if c in ["Численность населения всего", "СКР (всего)"]}

fig, axes = plt.subplots(1, len(wm_col_idx), figsize=(14, 5))
if len(wm_col_idx) == 1:
    axes = [axes]

for ax, (col, cidx) in zip(axes, wm_col_idx.items()):
    # Run 30 stochastic rollouts with MADDPG to get uncertainty bands
    sc_vals = []
    for sc in range(30):
        env_sc = KurskDemoEnv(world_ensemble, horizon=25, shock_std=CFG["shock_std"],
                              stochastic=True, seed=SEED+sc)
        obs_sc = env_sc.reset()
        done_sc = False
        while not done_sc:
            acts_sc = [maddpg_agents[i].act(obs_sc[i], explore=False) for i in range(N_AGENTS)]
            obs_sc, _, done_sc, info_sc = env_sc.step(acts_sc)
        sc_traj = env_sc.get_trajectory_df()
        if col in sc_traj.columns:
            sc_vals.append(sc_traj[col].values)

    if sc_vals:
        sc_arr  = np.array(sc_vals)           # (30, T)
        yr_arr  = sc_traj.index.values
        mu_sc   = sc_arr.mean(0)
        lo_sc   = np.percentile(sc_arr, 5,  axis=0)
        hi_sc   = np.percentile(sc_arr, 95, axis=0)

        ax.plot(HIST_DF.index, HIST_DF[col].values,
                color=HIST_COLOR, linewidth=2.5, label="История")
        ax.plot(yr_arr, mu_sc, color=MADDPG_COLOR, linewidth=2, label="MADDPG (среднее)")
        ax.fill_between(yr_arr, lo_sc, hi_sc, alpha=0.2, color=MADDPG_COLOR,
                        label="5–95 персентиль")
        ax.plot(baseline_traj.index, baseline_traj[col].values,
                color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="Базовый")
        ax.axvline(CFG["hist_end"], color="black", linewidth=1.2, linestyle="--", alpha=0.5)
        ax.set_title(col[:50], fontsize=9, fontweight="bold")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3); ax.tick_params(labelsize=7)

plt.suptitle("Неопределённость прогноза MADDPG (30 стохастических роллаутов)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "12_forecast_uncertainty_bands.png", dpi=150, bbox_inches="tight")
plt.close()
print("[Plot] 12_forecast_uncertainty_bands.png saved.")


SECTION 8 — VISUALISATIONS
[Plot] 06_trajectory_comparison_main.png saved.
[Plot] 07_effect_size_2050.png saved.
[Plot] 08_action_heatmap_maddpg.png saved.
[Plot] 09_action_outcome_correlations.png saved.
[Plot] 10_radar_kpi_2036.png saved.
[Plot] 11_reward_decomposition.png saved.
[Plot] 12_forecast_uncertainty_bands.png saved.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — EXPORT FOR SCIENTIFIC REPORT
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SECTION 9 — REPORT EXPORT")
print("="*70)

# ── 9.1 Summary statistics table ─────────────────────────────────────────────

KEY_METRICS = [
    "Численность населения всего",
    "СКР (всего)",
    "Коэффициент естественного прироста (на 1000)",
    "Коэффициент миграционного прироста (на 10000)",
    "Число абортов на 100 родов",
    "Число циклов ЭКО",
    "Обеспеченность местами в ДОУ (на 1000 детей)",
    "Общая площадь жилья на 1 жителя (кв. м)",
]

snapshot_years = [2025, 2030, 2036, 2040, 2050]
rows = []
for yr in snapshot_years:
    for col in KEY_METRICS:
        def _get_yr(traj, y, c):
            if c not in traj.columns:
                return np.nan
            if y in traj.index:
                return round(float(traj.loc[y, c]), 4)
            idx = abs(traj.index - y).argmin()
            return round(float(traj.iloc[idx][c]), 4)

        rows.append({
            "year":          yr,
            "metric":        col,
            "history_2025":  _get_yr(HIST_DF, 2025, col) if yr == 2025 else np.nan,
            "baseline":      _get_yr(baseline_traj, yr, col),
            "maddpg":        _get_yr(maddpg_traj, yr, col),
            "matd3":         _get_yr(matd3_traj, yr, col),
            "synthetic_fore":_get_yr(FORE_DF, yr, col),
        })

summary_table = pd.DataFrame(rows)
summary_table["maddpg_delta_pct"] = (
    (summary_table["maddpg"] - summary_table["baseline"])
    / summary_table["baseline"].abs().replace(0, np.nan)
    * 100
).round(2)
summary_table["matd3_delta_pct"] = (
    (summary_table["matd3"] - summary_table["baseline"])
    / summary_table["baseline"].abs().replace(0, np.nan)
    * 100
).round(2)

summary_table.to_csv(REPORT_DIR / "summary_kpi_table_all_scenarios.csv", index=False)
print("[Report] summary_kpi_table_all_scenarios.csv saved.")

# ── 9.2 Agent action priority ranking ────────────────────────────────────────
# Rank each agent's actions by their mean absolute value (most used = most important)
action_priority = (
    maddpg_action_log
    .groupby(["agent_name", "action_name"])["action_value"]
    .agg(mean_abs=lambda x: x.abs().mean(), mean=np.mean, std=np.std)
    .reset_index()
    .sort_values(["agent_name", "mean_abs"], ascending=[True, False])
)
action_priority.to_csv(REPORT_DIR / "maddpg_action_priority_ranking.csv", index=False)
print("[Report] maddpg_action_priority_ranking.csv saved.")

# ── 9.3 Infrastructure actions → demographic improvement table ────────────────
infra_agents = ["RegionalGov", "HealthcareSystem", "EducationInfra", "MigrationPolicy"]
infra_impact = corr_tfr[corr_tfr["agent"].isin(infra_agents)].copy()
infra_impact = pd.concat([
    infra_impact,
    corr_mig[corr_mig["agent"].isin(infra_agents)],
    corr_pop[corr_pop["agent"].isin(infra_agents)],
], ignore_index=True)
infra_impact.to_csv(REPORT_DIR / "infra_action_demo_impact.csv", index=False)
print("[Report] infra_action_demo_impact.csv saved.")

# ── 9.4 Full experiment metadata ──────────────────────────────────────────────
metadata = {
    "experiment_id": f"kursk_stage1_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "config":        CFG,
    "agents":        [
        {k: v for k, v in spec.items() if k not in ["obs_indices"]}
        for spec in AGENT_SPECS
    ],
    "world_model": {
        "ensemble_size":   ENSEMBLE_SIZE,
        "train_epochs":    WORLD_EPOCHS,
        "best_val_nll":    best_val_loss,
        "state_dim":       STATE_DIM,
        "action_dim":      ACTION_DIM_T,
    },
    "results_2050": {
        "median_maddpg_effect_pct": round(
            float(effect_df["MADDPG_Δ%_2050"].dropna().median()), 3),
        "median_matd3_effect_pct": round(
            float(effect_df["MATD3_Δ%_2050"].dropna().median()), 3),
    },
}
with open(REPORT_DIR / "experiment_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("[Report] experiment_metadata.json saved.")

# ── 9.5 List all output files ─────────────────────────────────────────────────
print("\n" + "="*70)
print("OUTPUT FILES")
print("="*70)

all_files = sorted([p for p in BASE_DIR.rglob("*") if p.is_file()])
for f in all_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.relative_to(BASE_DIR)}  [{size_kb:.1f} KB]")

print("\n" + "="*70)
print("STAGE 1 COMPLETE")
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"All outputs saved under: {BASE_DIR.resolve()}")
print("="*70)


SECTION 9 — REPORT EXPORT
[Report] summary_kpi_table_all_scenarios.csv saved.
[Report] maddpg_action_priority_ranking.csv saved.
[Report] infra_action_demo_impact.csv saved.
[Report] experiment_metadata.json saved.

OUTPUT FILES
  models/MADDPG_EducationInfra_actor_ep500.pt  [294.7 KB]
  models/MADDPG_EducationInfra_actor_ep600.pt  [294.7 KB]
  models/MADDPG_Families_actor_ep500.pt  [297.6 KB]
  models/MADDPG_Families_actor_ep600.pt  [297.6 KB]
  models/MADDPG_HealthcareSystem_actor_ep500.pt  [296.7 KB]
  models/MADDPG_HealthcareSystem_actor_ep600.pt  [296.7 KB]
  models/MADDPG_Households_actor_ep500.pt  [294.6 KB]
  models/MADDPG_Households_actor_ep600.pt  [294.6 KB]
  models/MADDPG_Migrants_actor_ep500.pt  [282.6 KB]
  models/MADDPG_Migrants_actor_ep600.pt  [282.6 KB]
  models/MADDPG_MigrationPolicy_actor_ep500.pt  [282.7 KB]
  models/MADDPG_MigrationPolicy_actor_ep600.pt  [282.7 KB]
  models/MADDPG_RegionalGov_actor_ep500.pt  [304.6 KB]
  models/MADDPG_RegionalGov_actor_ep600.pt  [

In [ ]:
import shutil
from pathlib import Path

base_dir = Path("/content/kursk_marl_output")
zip_path = "/content/kursk_marl_output.zip"

if not base_dir.exists():
    raise FileNotFoundError(f"{base_dir} not found. Нечего архивировать.")

shutil.make_archive("/content/kursk_marl_output", "zip", base_dir)
print(f"Архив создан: {zip_path}")

Архив создан: /content/kursk_marl_output.zip
